# Reproject GeoTIFFs for the web map

Reprojects the SERC rasters to EPSG:3857 and writes Cloud Optimized GeoTIFFs next to this notebook.
`maplibre-cog-protocol` (see `src/composables/useGeotiff.ts`) only reads EPSG:3857 COGs.
Code is adapted from `forestgeo/other-data/soil.ipynb`.

In [ ]:
import numpy as np
import rasterio as rio
from rasterio import warp
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

DATA = "~/Dropbox/cosmicds/forestgeo/other-data"  # source rasters
DST_CRS = "EPSG:3857"  # what the web map (and the cog:// protocol) needs

In [ ]:
def reproject(src_path, dst_path, dst_crs=DST_CRS, nodata=None, res=None):
    """Reproject band 1 of src_path to dst_crs and write it as a float32 COG.

    nodata overrides the source nodata (the LiDAR DEM uses 0 but doesn't declare it).
    res is the output pixel size in dst_crs units (None keeps roughly the source resolution).
    """
    with rio.open(src_path) as src:
        src_nodata = src.nodata if nodata is None else nodata
        dst_transform, dst_w, dst_h = warp.calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=res
        )
        dst = np.full((dst_h, dst_w), np.nan, dtype="float32")
        warp.reproject(
            source=src.read(1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )

    # NaN is the nodata value, so it is declared in the file and renders transparent
    with rio.open(
        dst_path, "w", driver="COG",
        width=dst_w, height=dst_h, count=1, dtype="float32",
        crs=dst_crs, transform=dst_transform, nodata=np.nan,
        compress="DEFLATE", predictor=3, blocksize=256,
    ) as out:
        out.write(dst, 1)
    return dst

In [ ]:
import os
DATA = os.path.expanduser(DATA)

# 1 m on the ground is ~1.29 m in Web Mercator at this latitude (1 / cos 38.9°)
dem = reproject(f"{DATA}/SERC_MDLiDAR_m_ForestGEO_plot/SERC_MDLiDAR_m_ForestGEO_plot.tif",
                "serc_dem_3857.tif", nodata=0, res=1.29)
soil = reproject(f"{DATA}/SERC_soil_moisture_2012/SERC_soil_moisture_2012.tif",
                 "serc_soil_moisture_2012_3857.tif", res=1.29)

In [ ]:
# sanity check: the written files, their CRS/bounds/nodata, and a quick look
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
for ax, path, cmap in zip(axs, ["serc_dem_3857.tif", "serc_soil_moisture_2012_3857.tif"], ["terrain", "Blues"]):
    with rio.open(path) as f:
        print(path, f.crs, f.shape, f.nodata, f.bounds, f.overviews(1))
        a = f.read(1, masked=True)
        print(f"  min {a.min():.2f} max {a.max():.2f}")
        ax.imshow(a, cmap=cmap)
        ax.set_title(path)